# Mi8 Multipath — Data Parser

Parses all six Mi8 GnssLog sessions and uses the device's own `MultipathIndicator` field (0 = clean, 1 = multipath) as the ground-truth label. No SPAN time-sync is needed.

**Input:** `data/01_raw/20*/Mi8_GnssLog.txt` (6 sessions)

**Output:** `data/02_interim/mi8_epochs.csv`

## 1. Setup & Configuration

In [ ]:
import os
import glob
import pandas as pd
import numpy as np

BASE_DIR   = os.path.abspath(os.path.join(os.getcwd(), '../..'))
RAW_DIR    = os.path.join(BASE_DIR, 'data/01_raw')
OUTPUT_DIR = os.path.join(BASE_DIR, 'data/02_interim')
OUTPUT_CSV = os.path.join(OUTPUT_DIR, 'mi8_epochs.csv')
os.makedirs(OUTPUT_DIR, exist_ok=True)

MI8_LOGS = sorted(glob.glob(os.path.join(RAW_DIR, '*', 'Mi8_GnssLog.txt')))
print(f'Found {len(MI8_LOGS)} Mi8 log files:')
for p in MI8_LOGS:
    print(' ', os.path.relpath(p, RAW_DIR))

## 2. Parse Raw Mi8 GNSS Logs

Extracts all `Raw` measurement rows from each session. The Mi8 logs contain a richer column set than the Pixel devices, including pseudorange rate, constellation type, and baseband C/N0.

In [ ]:
REQUIRED_COLS = [
    'TimeNanos', 'FullBiasNanos', 'State',
    'ReceivedSvTimeUncertaintyNanos', 'Cn0DbHz',
    'PseudorangeRateMetersPerSecond', 'PseudorangeRateUncertaintyMetersPerSecond',
    'AccumulatedDeltaRangeState', 'AccumulatedDeltaRangeMeters',
    'AccumulatedDeltaRangeUncertaintyMeters',
    'CarrierFrequencyHz', 'MultipathIndicator',
    'SnrInDb', 'ConstellationType', 'AgcDb', 'BasebandCn0DbHz',
]

def parse_mi8_log(file_path: str, session_label: str) -> pd.DataFrame:
    with open(file_path, 'r') as fh:
        lines = fh.readlines()

    raw_lines  = [l.strip().split(',') for l in lines if l.startswith('Raw')]
    header_str = next((l for l in lines if l.startswith('# Raw')), None)
    if not header_str or not raw_lines:
        print(f'  WARNING: no Raw rows in {file_path}')
        return pd.DataFrame()

    cols = header_str.strip().replace('# Raw,', '').split(',')
    cols.insert(0, 'Raw')
    df = pd.DataFrame(raw_lines, columns=cols)

    available = [c for c in REQUIRED_COLS if c in df.columns]
    missing   = [c for c in REQUIRED_COLS if c not in df.columns]
    if missing:
        print(f'  NOTE: columns not found in {session_label}: {missing}')

    df = df[available].copy()
    for col in df.columns:
        if col in ('TimeNanos', 'FullBiasNanos'):
            df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')
        else:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    df['session'] = session_label
    return df

frames = []
for path in MI8_LOGS:
    label = os.path.basename(os.path.dirname(path))
    df = parse_mi8_log(path, label)
    if not df.empty:
        frames.append(df)
        print(f'  {label}: {len(df):,} raw rows')

raw_df = pd.concat(frames, ignore_index=True)
print(f'\nTotal raw rows across all sessions: {len(raw_df):,}')
raw_df.head()

## 3. Outlier Rejection

Applies Google's official quality filters: valid `FullBiasNanos`, positive `TimeNanos`, TOW decoded/known state flag, and timing uncertainty ≤ 500 ns.

In [ ]:
def apply_outlier_rejection(df: pd.DataFrame) -> pd.DataFrame:
    n0 = len(df)
    df = df.dropna(subset=['FullBiasNanos'])
    df = df[df['FullBiasNanos'] != 0]
    df = df.dropna(subset=['TimeNanos'])
    df = df[df['TimeNanos'] > 0]
    state_ok = ((df['State'].astype(int) & (1 << 3)) != 0) | \
               ((df['State'].astype(int) & (1 << 14)) != 0)
    df = df[state_ok]
    df = df[df['ReceivedSvTimeUncertaintyNanos'] <= 500]
    print(f'Rejected {n0 - len(df):,} rows  |  Kept {len(df):,} rows')
    return df

clean_df = apply_outlier_rejection(raw_df.copy())
clean_df.head()

## 4. Compute GPS Time of Week

Derives `GpsTimeNanos` (nanoseconds since the start of the current GPS week) from `TimeNanos` and `FullBiasNanos`. This provides an absolute time reference for ordering and cross-session analysis.

In [ ]:
NANOS_PER_SECOND = 1e9
SECONDS_PER_WEEK = 604800

raw_gps_ns = clean_df['TimeNanos'].astype(np.int64) - clean_df['FullBiasNanos'].astype(np.int64)
clean_df['GpsTimeNanos'] = raw_gps_ns % int(SECONDS_PER_WEEK * NANOS_PER_SECOND)

print('GpsTimeNanos range:')
print(f'  min = {clean_df["GpsTimeNanos"].min():,}  max = {clean_df["GpsTimeNanos"].max():,}')
clean_df[['session', 'GpsTimeNanos', 'Cn0DbHz', 'MultipathIndicator']].head(10)

## 5. Label Distribution

The Mi8 is the only device in this dataset that actively reports `MultipathIndicator = 1`. These are real hardware-detected multipath events, making this the cleanest possible label source.

In [ ]:
total = len(clean_df)
dist  = clean_df['MultipathIndicator'].value_counts()
print('=== Overall MultipathIndicator distribution ===')
for val, cnt in dist.items():
    print(f'  {int(val)} : {cnt:>7,}  ({100*cnt/total:.1f}%)')

print('\n=== Per-session breakdown ===')
session_dist = (
    clean_df.groupby(['session', 'MultipathIndicator'])
    .size()
    .unstack(fill_value=0)
    .rename(columns={0.0: 'Clean', 1.0: 'Multipath'})
)
if 'Multipath' in session_dist.columns:
    session_dist['Multipath_%'] = (
        100 * session_dist['Multipath'] / session_dist.sum(axis=1)
    ).round(1)
print(session_dist)

## 6. Export

Drops the raw time columns that are not predictive features and saves the final dataset.

In [ ]:
final_df = clean_df.drop(columns=['TimeNanos', 'FullBiasNanos'], errors='ignore').copy()
final_df.dropna(subset=['MultipathIndicator'], inplace=True)
final_df['MultipathIndicator'] = final_df['MultipathIndicator'].astype(int)

final_df.to_csv(OUTPUT_CSV, index=False)
print(f'Saved {len(final_df):,} rows to: {OUTPUT_CSV}')
print(f'Columns: {list(final_df.columns)}')
final_df.head()